# Model training

In this notebook, we will finally train our model!

## Imports

Let's start by importing the necessary libraries. For training the model, we will use PyTorch, one of the most popular AI libraries in the world.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Training device

In this cell we check if this computer supports hardware accelerated AI training. If yes, we use it, if not, we use the CPU.

In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f"Using GPU")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
  print(f"Using Apple NPU")
else:
  device = torch.device("cpu")
  print(f"Using CPU")

## Preprocessing

Before training the model, we need to format the images correctly.

We also do something clever - we make it so that every time our model sees an image, it will be randomly rotated by 15 degrees. This way the model won't try to memorize the exact images, but look for more general patterns (i.e. it will not memorize a box in one specific orientation, but will learn how boxes look in general). This technique is called data augmentation.

In [ ]:
train_transforms = transforms.Compose(
  [
    transforms.Resize((288, 288)),  # Make sure all images are the same size
    transforms.RandomRotation(degrees=15),  # Add some random rotation for data augmentation
    transforms.ToTensor(),  # Convert the images to PyTorch format
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize the pixel values to be between -1 and 1
  ]
)

# We want to test on the original photos without augmentation, so we only resize and normalize
test_transforms = transforms.Compose(
  [
    transforms.Resize((288, 288)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
  ]
)

## Train/test split

Now we split the images into training and testing set. We will use the training set for training the model and testing set for checking how well it works.

In [ ]:
# Dataset with train preprocessing
full_dataset_for_train = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=train_transforms
)

# Dataset with test preprocessing
full_dataset_for_test = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=test_transforms
)

# We want to split the dataset, so that proportions of forward/obstacle photos are the same in the train and test sets

idxs = np.arange(len(full_dataset_for_train))  # Get the indices of all samples in the dataset
labels = full_dataset_for_train.targets  # Get the labels for each sample (0 for forward, 1 for obstacle)

train_idx, test_idx = train_test_split(
  idxs,
  test_size=0.2,  # Use 20% of the data for testing
  stratify=labels  # Ensure that the class distribution is the same in both sets
)

# Create the train and test datasets using the respective indices
train_dataset = torch.utils.data.Subset(full_dataset_for_train, train_idx)
test_dataset = torch.utils.data.Subset(full_dataset_for_test, test_idx)

# Create data loaders for the train and test datasets
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
print(f"Forward photos: {np.sum(np.array(labels)[train_idx] == 0)}")
print(f"Obstacle photos: {np.sum(np.array(labels)[train_idx] == 1)}")

In [ ]:
print(f"Forward percentage: {np.sum(np.array(labels) == 0) / len(labels) * 100:.2f}%")
print(f"Obstacle percentage: {np.sum(np.array(labels) == 1) / len(labels) * 100:.2f}%")

## Model architecture

Now we define our model architecture!

We mostly use three types of layers:
- `Conv2d` - this applies the convolution explained during the lecture
- `MaxPool2d` - replaces groups of pixels with their maximum values, making the image smaller
- `BatchNorm2d` - this does some fancy math stuff, so that our training is more stable ;)
- `ReLU` - this applies the ReLU(val) = max(0, val)

At the end we also use:
- `AdaptiveAvgPool2d` - takes the average of entire image layers (channels)
- `Flatten` - formats our data for the `Linear` layer (converts the data to 1D array)
- `Linear` - regular fully-connected neural network layer
- `Dropout` - randomly deactivates some neurons during training, so that the network must learn how to act without them (improves stability, reduces overfitting)

In [ ]:
model = torch.hub.load(
    'lukemelas/EfficientNet-PyTorch', 
    'efficientnet_b0', 
    pretrained=False, # False because you are using your own weights
    num_classes=1    # Change this to your actual number of classes
)

In [ ]:
model

In [ ]:
for param in model.parameters():
  param.requires_grad = False

in_features = model._fc.in_features
model._fc = nn.Linear(in_features, 1)

model = model.to(device)

## Learning parameters

In [ ]:
# This function tells the model how to measure it's mistakes (loss). If the |(actual label) - (model prediction)| is large, the loss will be large
criterion = nn.BCEWithLogitsLoss()

# This function tells the model how to update itself based on the mistakes it made. It will try to update itself in a way that reduces the loss
optimizer = optim.Adam(model._fc.parameters(), lr=0.001)

# How many times the model will go through the entire training dataset, trying to correct itself
n_epochs = 20

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, n_epochs, best_eval_loss, checkpoint_path, phase_name="Train"):
  for epoch in range(n_epochs):
    running_loss = 0.0

    model.train()  # Some layers behave differently during training and evaluation

    for data in train_loader:
      inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

      # We tell the model to forget previous gradients and start counting again
      optimizer.zero_grad()

      # We get model predictions and compare them to actual labels
      outputs = model(inputs)
      loss = criterion(outputs, labels)

      # Update model weights to reduce the loss
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

    model.eval()
    with torch.no_grad():
      eval_loss = 0.0

      for data in test_loader:
        inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        eval_loss += loss.item()

    print(
      f"{phase_name} Epoch {epoch+1}/{n_epochs}, "
      f"Loss: {running_loss/len(train_loader):.4f}, "
      f"Eval Loss: {eval_loss/len(test_loader):.4f}"
    )

    # Save best model across all phases
    if eval_loss < best_eval_loss:
      best_eval_loss = eval_loss
      torch.save(model.state_dict(), checkpoint_path)

  return best_eval_loss


best_eval_loss = float("inf")
best_eval_loss = train_model(
  model=model,
  train_loader=train_loader,
  test_loader=test_loader,
  criterion=criterion,
  optimizer=optimizer,
  n_epochs=n_epochs,
  best_eval_loss=best_eval_loss,
  checkpoint_path="best_model_with_backbone.pth",
  phase_name="Head training"
)

In [ ]:
# Fine-tuning: unfreeze the full backbone and train with a low learning rate
for param in model.parameters():
  param.requires_grad = True

fine_tune_lr = 1e-5
fine_tune_epochs = 5
optimizer = optim.Adam(model.parameters(), lr=fine_tune_lr)

best_eval_loss = train_model(
  model=model,
  train_loader=train_loader,
  test_loader=test_loader,
  criterion=criterion,
  optimizer=optimizer,
  n_epochs=fine_tune_epochs,
  best_eval_loss=best_eval_loss,
  checkpoint_path="best_model_with_backbone.pth",
  phase_name="Fine-tune"
)

## Evaluation

In [ ]:
all_predictions = []
all_labels = []

# Restore the best model
model.load_state_dict(torch.load("best_model_with_backbone.pth"))

model.eval()

# Check how well the model is doing on the test set
with torch.no_grad():
  for data in test_loader:
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)
    outputs = model(inputs)
    predictions = torch.sigmoid(outputs) > 0.5
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

all_predictions = np.array(all_predictions).flatten()
all_labels = np.array(all_labels).flatten()

# Calculate evaluation metrics
accuracy = np.sum(all_predictions == all_labels.astype(bool)) / len(all_labels)
precision = np.sum((all_predictions == 1) & (all_labels == 1)) / np.sum(all_predictions == 1)
recall = np.sum((all_predictions == 1) & (all_labels == 1)) / np.sum(all_labels == 1)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Metrics meaning:
- accuracy - how many times the model was correct
- precision - out of all the images the model classified as obstacles, how many were actual obstacles
- recall - out of all the images that were real obstacles, how many did the model find

If precision is low, the robot will often turn even if the road is clear.
If recall is low, the robot will bump into obstacles.

**The model is now ready for testing on the robot!**